# Tool Use in the Agent Loop

This notebook goes deeper on **tool use**, the mechanism that lets an agent interact with external systems, retrieve live data, and perform actions beyond what the LLM knows.

**Use case: Procurement reorder decision**

*Should we reorder part SKU-8821 (high-tensile flange bearings)?*

The agent must check current inventory, pull an 8-week demand forecast, fetch quotes from two competing suppliers, calculate the reorder point with a safety buffer, and produce a concrete recommendation with supplier selection rationale.

Procurement decisions depend on multiple data sources that must be combined before a recommendation can be made. No single tool call is sufficient, the agent must orchestrate a sequence of lookups and a calculation.


We will
- Understand what a tool *is* at the implementation level (a callable + a schema)
- See how multiple tools collaborate within a single agent loop run
- Understand how tool results accumulate and inform subsequent decisions
- Distinguish tools that *retrieve* data from tools that *compute*


## Setup

In [1]:
import os
import google.genai as genai

client = genai.Client() # api key in .env (loaded)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


---
## The Task

A procurement analyst has flagged SKU-8821 for review. The agent must gather all
necessary data and deliver a reorder recommendation with quantitative justification.


In [2]:
TASK = (
    "Determine whether we should place a reorder for SKU-8821 (high-tensile flange bearings). "
    "Check current inventory and available stock. Get the 8-week demand forecast. "
    "Fetch quotes from both SUPP-A and SUPP-B. "
    "Calculate the reorder point using a 10% safety buffer (0.5 weeks of safety stock). "
    "Recommend: (1) should we reorder now? (2) if yes, which supplier and how many units?"
)
print("Task:", TASK)


Task: Determine whether we should place a reorder for SKU-8821 (high-tensile flange bearings). Check current inventory and available stock. Get the 8-week demand forecast. Fetch quotes from both SUPP-A and SUPP-B. Calculate the reorder point using a 10% safety buffer (0.5 weeks of safety stock). Recommend: (1) should we reorder now? (2) if yes, which supplier and how many units?


---
## Tool Implementations

Each tool is a Python function that simulates a backend system call.
In production these would query an ERP (e.g. SAP, Oracle), a demand planning system,
and a supplier portal.

Notice the four distinct tool types in this set:
- **Inventory lookup**: reads a current system state
- **Forecast retrieval**: reads a time-series prediction
- **External quote fetch**: queries a supplier-specific system (called twice, once per supplier)
- **Calculation**: deterministic arithmetic; no external call needed


In [3]:
#  Tool implementations

def get_inventory_status(sku: str) -> str:
    # Query current inventory levels for a given SKU.
    data = {
        "SKU-8821": (
            "SKU: SKU-8821 | Description: High-tensile flange bearings M20 | "
            "On-hand: 847 units | Committed to open orders: 620 units | "
            "Available (uncommitted): 227 units | Unit: EA | "
            "Warehouse: NL-04 | Last replenishment: 2024-09-12"
        )
    }
    return data.get(sku.upper(), f"SKU '{sku}' not found in inventory system.")


def get_demand_forecast(sku: str, weeks: int) -> str:
    # Return the weekly demand forecast for the next N weeks.
    weeks = int(weeks)  # LLM may pass a float; coerce to int for slicing
    if sku.upper() == "SKU-8821":
        weekly = [182, 197, 214, 188, 223, 207, 184, 218]
        subset = weekly[:min(weeks, 8)]
        total  = sum(subset)
        avg    = total / len(subset)
        return (
            f"SKU-8821 demand forecast (next {len(subset)} weeks): {subset} | "
            f"Total: {total} units | Avg/week: {avg:.1f} units | "
            f"Trend: slightly upward (new production line at customer IMR-DE coming online W+3)"
        )
    return f"No forecast data for '{sku}'"


def get_supplier_quote(sku: str, supplier_id: str) -> str:
    # Fetch quote from a specific supplier for a given SKU.
    quotes = {
        ("SKU-8821", "SUPP-A"): (
            "Supplier: PrecisionParts GmbH (SUPP-A) | "
            "Unit price: EUR 14.20 | Lead time: 3 weeks | "
            "Min order qty: 500 units | Reliability score: 9.2/10 | "
            "On-time delivery rate (12mo): 96.4% | ISO 9001 certified"
        ),
        ("SKU-8821", "SUPP-B"): (
            "Supplier: TechBearings Asia Ltd (SUPP-B) | "
            "Unit price: EUR 11.80 | Lead time: 6 weeks | "
            "Min order qty: 1000 units | Reliability score: 7.8/10 | "
            "On-time delivery rate (12mo): 81.2% | One quality incident (Q2 2024)"
        ),
    }
    key = (sku.upper(), supplier_id.upper())
    return quotes.get(key, f"No quote found for SKU '{sku}' from supplier '{supplier_id}'")


def calculate_reorder_point(
    avg_weekly_demand: float,
    lead_time_weeks: float,
    safety_weeks: float
) -> str:
    # Calculate reorder point: (demand * lead_time) + (demand * safety_buffer)
    cycle_stock  = avg_weekly_demand * lead_time_weeks
    safety_stock = avg_weekly_demand * safety_weeks
    rop          = cycle_stock + safety_stock
    return (
        f"Reorder point: {rop:.0f} units | "
        f"Cycle stock: {cycle_stock:.0f} (avg {avg_weekly_demand}/wk x {lead_time_weeks}wk lead time) | "
        f"Safety stock: {safety_stock:.0f} (avg {avg_weekly_demand}/wk x {safety_weeks}wk buffer)"
    )


TOOLS = {
    "get_inventory_status":    get_inventory_status,
    "get_demand_forecast":      get_demand_forecast,
    "get_supplier_quote":       get_supplier_quote,
    "calculate_reorder_point":  calculate_reorder_point,
}

# Sanity-check each tool
print(get_inventory_status("SKU-8821"))
print()
print(get_demand_forecast("SKU-8821", 8))
print()
print(get_supplier_quote("SKU-8821", "SUPP-A"))
print()
print(calculate_reorder_point(201.6, 3, 0.5))

SKU: SKU-8821 | Description: High-tensile flange bearings M20 | On-hand: 847 units | Committed to open orders: 620 units | Available (uncommitted): 227 units | Unit: EA | Warehouse: NL-04 | Last replenishment: 2024-09-12

SKU-8821 demand forecast (next 8 weeks): [182, 197, 214, 188, 223, 207, 184, 218] | Total: 1613 units | Avg/week: 201.6 units | Trend: slightly upward (new production line at customer IMR-DE coming online W+3)

Supplier: PrecisionParts GmbH (SUPP-A) | Unit price: EUR 14.20 | Lead time: 3 weeks | Min order qty: 500 units | Reliability score: 9.2/10 | On-time delivery rate (12mo): 96.4% | ISO 9001 certified

Reorder point: 706 units | Cycle stock: 605 (avg 201.6/wk x 3wk lead time) | Safety stock: 101 (avg 201.6/wk x 0.5wk buffer)


---
## Tool Schemas (Gemini `FunctionDeclaration`)

Each schema tells the LLM:
- the tool's **name** and **purpose** (description)
- what **arguments** it expects (parameter names, types, descriptions)
- which arguments are **required**

A well-written description is as important as the schema structure.
The LLM decides *which* tool to call and *when* based on the description alone.


In [9]:
from google.genai import types

TOOL_DEFINITIONS = {
    "function_declarations": [

        {
            "name": "get_inventory_status",
            "description": (
                "Returns current inventory for a SKU: on-hand quantity, committed units, "
                "and available (uncommitted) stock. Call this first to understand the current position."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "sku": {
                        "type": "string",
                        "description": "The SKU code, e.g. SKU-8821."
                    }
                },
                "required": ["sku"]
            }
        },

        {
            "name": "get_demand_forecast",
            "description": (
                "Returns the weekly demand forecast for a SKU over a specified horizon. "
                "Use 8 weeks for a standard procurement horizon."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "sku": {
                        "type": "string",
                        "description": "The SKU code."
                    },
                    "weeks": {
                        "type": "integer",
                        "description": "Number of weeks to forecast (1-8)."
                    }
                },
                "required": ["sku", "weeks"]
            }
        },

        {
            "name": "get_supplier_quote",
            "description": (
                "Fetch a supplier quote for a SKU: unit price, lead time, minimum order quantity, "
                "and reliability score. Call once per supplier. "
                "Available suppliers for SKU-8821: SUPP-A, SUPP-B."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "sku": {
                        "type": "string",
                        "description": "The SKU code."
                    },
                    "supplier_id": {
                        "type": "string",
                        "description": "Supplier identifier, e.g. SUPP-A or SUPP-B."
                    }
                },
                "required": ["sku", "supplier_id"]
            }
        },

        {
            "name": "calculate_reorder_point",
            "description": (
                "Calculate the reorder point: the inventory level at which a new order must be placed. "
                "Formula: (avg_weekly_demand * lead_time_weeks) + (avg_weekly_demand * safety_weeks). "
                "Use safety_weeks=0.5 for a 10% buffer."
            ),
            "parameters": {
                "type": "object",
                "properties": {
                    "avg_weekly_demand": {
                        "type": "number",
                        "description": "Average weekly demand in units."
                    },
                    "lead_time_weeks": {
                        "type": "number",
                        "description": "Supplier lead time in weeks."
                    },
                    "safety_weeks": {
                        "type": "number",
                        "description": "Safety buffer in weeks, e.g. 0.5."
                    }
                },
                "required": [
                    "avg_weekly_demand",
                    "lead_time_weeks",
                    "safety_weeks"
                ]
            }
        },

    ]
}

In [11]:
def model(msg):
    return client.models.generate_content(model='gemini-2.5-flash-lite',
                                              contents=msg,
                                              config = types.GenerateContentConfig(
                                                  system_instruction=(
                                                        "You are a procurement analyst assistant. "
                                                        "Use the tools to gather inventory, demand, and supplier data. "
                                                        "Calculate the reorder point and provide a concrete recommendation: "
                                                        "reorder or not, preferred supplier, and suggested order quantity."
                                                    ),
                                                  tools=[TOOL_DEFINITIONS])
                            )

print("Model ready with", len(TOOL_DEFINITIONS['function_declarations']), "tools.")

Model ready with 4 tools.


---
## State and Loop Functions

These are identical to the pattern from Session 2. The only thing that changes
between agent implementations is the **tools** and the **task**.
The loop mechanics are reusable.


In [14]:
class AgentState:
    def __init__(self, goal: str, max_steps: int = 12):
        self.goal         = goal
        self.messages     = []
        self.step_count   = 0
        self.max_steps    = max_steps
        self.done         = False
        self.final_answer = None

def observe(state):
    if not state.messages:
        state.messages.append({"role": "user", "parts": [{"text": state.goal}]})

def reason(state):
    print(f"\n[Step {state.step_count + 1}] Calling LLM...")
    return model(state.messages)

def decide(response):
    parts = response.candidates[0].content.parts
    tool_calls = []
    for p in parts:
        func = getattr(p, "function_call", None)
        if func and getattr(func, "name", None):
            tool_calls.append({"name": func.name, "args": dict(func.args)})
    return ("tool_calls", tool_calls) if tool_calls else ("final_answer", response.text)

def act(tool_calls):
    results = []
    for tc in tool_calls:
        name, args = tc["name"], tc["args"]
        result = TOOLS[name](**args) if name in TOOLS else f"Error: unknown tool '{name}'"
        print(f"  [{name}] {args}")
        print(f"    => {result[:100]}")
        results.append({"name": name, "result": result})
    return results

def update(state, response, tool_results):
    state.messages.append(response.candidates[0].content)
    state.messages.append({"role": "user", "parts": [
        {"function_response": {"name": tr["name"], "response": {"result": tr["result"]}}}
        for tr in tool_results
    ]})
    state.step_count += 1

def run_agent(state):
    print("=" * 65)
    print(f"GOAL: {state.goal[:100]}...")
    print("=" * 65)
    observe(state)
    while not state.done:
        if state.step_count >= state.max_steps:
            state.final_answer = "Max steps reached."
            state.done = True
            break
        response = reason(state)
        dtype, dval = decide(response)
        if dtype == "final_answer":
            state.final_answer = dval
            state.done = True
        else:
            tool_results = act(dval)
            update(state, response, tool_results)
    print("\n" + "=" * 65)
    print("RECOMMENDATION:")
    print(state.final_answer)
    print("=" * 65)
    return state.final_answer


---
## Run the Procurement Agent

Watch how the agent sequences its tool calls:
1. First it checks inventory to understand the current position
2. Then gets the demand forecast to quantify consumption rate
3. Fetches quotes from both suppliers
4. Calculates the reorder point using the relevant supplier's lead time
5. Combines all data into a recommendation

Notice that the agent may call `calculate_reorder_point` twice(once per supplier) to compare how the different lead times affect the reorder trigger.


In [15]:
state = AgentState(goal=TASK)
recommendation = run_agent(state)

GOAL: Determine whether we should place a reorder for SKU-8821 (high-tensile flange bearings). Check curre...

[Step 1] Calling LLM...
  [get_inventory_status] {'sku': 'SKU-8821'}
    => SKU: SKU-8821 | Description: High-tensile flange bearings M20 | On-hand: 847 units | Committed to op
  [get_demand_forecast] {'sku': 'SKU-8821', 'weeks': 8}
    => SKU-8821 demand forecast (next 8 weeks): [182, 197, 214, 188, 223, 207, 184, 218] | Total: 1613 unit
  [get_supplier_quote] {'sku': 'SKU-8821', 'supplier_id': 'SUPP-A'}
    => Supplier: PrecisionParts GmbH (SUPP-A) | Unit price: EUR 14.20 | Lead time: 3 weeks | Min order qty:
  [get_supplier_quote] {'sku': 'SKU-8821', 'supplier_id': 'SUPP-B'}
    => Supplier: TechBearings Asia Ltd (SUPP-B) | Unit price: EUR 11.80 | Lead time: 6 weeks | Min order qt

[Step 2] Calling LLM...
  [calculate_reorder_point] {'lead_time_weeks': 3, 'safety_weeks': 0.5, 'avg_weekly_demand': 201.6}
    => Reorder point: 706 units | Cycle stock: 605 (avg 201.6/wk x 3wk 

---
## What the Agent Sees

After the run, we can trace exactly what information the agent had access to
at each decision point. This is the agent's full working memory.


In [16]:
print(f"Total LLM calls: {state.step_count}")
print(f"Total messages:  {len(state.messages)}")
print()
def decide(response):
    parts = response.candidates[0].content.parts
    tool_calls = []
    for p in parts:
        func = getattr(p, "function_call", None)
        if func and getattr(func, "name", None):
            tool_calls.append({"name": func.name, "args": dict(func.args)})
    return ("tool_calls", tool_calls) if tool_calls else ("final_answer", response.text)
for i, msg in enumerate(state.messages):
    if isinstance(msg, dict):
        role, parts = msg["role"].upper(), msg["parts"]
    else:
        role, parts = msg.role.upper(), msg.parts
    for part in parts:
        if isinstance(part, dict):
            if "text" in part:
                print(f"[{i}] {role}: {part['text'][:120]}")
            elif "function_response" in part:
                fr = part["function_response"]
                print(f"[{i}] {role} result: {fr['name']}() => {str(fr['response'])[:80]}")
        else:
            if hasattr(part, "text") and part.text:
                print(f"[{i}] {role}: {part.text[:120]}")
            elif hasattr(part, "function_call") and part.function_call.name:
                print(f"[{i}] {role} call: {part.function_call.name}({dict(part.function_call.args)})")
    print()


Total LLM calls: 2
Total messages:  5

[0] USER: Determine whether we should place a reorder for SKU-8821 (high-tensile flange bearings). Check current inventory and ava

[1] MODEL call: get_inventory_status({'sku': 'SKU-8821'})
[1] MODEL call: get_demand_forecast({'sku': 'SKU-8821', 'weeks': 8})
[1] MODEL call: get_supplier_quote({'sku': 'SKU-8821', 'supplier_id': 'SUPP-A'})
[1] MODEL call: get_supplier_quote({'sku': 'SKU-8821', 'supplier_id': 'SUPP-B'})

[2] USER result: get_inventory_status() => {'result': 'SKU: SKU-8821 | Description: High-tensile flange bearings M20 | On-h
[2] USER result: get_demand_forecast() => {'result': 'SKU-8821 demand forecast (next 8 weeks): [182, 197, 214, 188, 223, 2
[2] USER result: get_supplier_quote() => {'result': 'Supplier: PrecisionParts GmbH (SUPP-A) | Unit price: EUR 14.20 | Lea
[2] USER result: get_supplier_quote() => {'result': 'Supplier: TechBearings Asia Ltd (SUPP-B) | Unit price: EUR 11.80 | L

[3] MODEL call: calculate_reorder_point({'lead_

To summarise,
1. **A tool is a callable + a schema.** The Python function handles execution;
   the `FunctionDeclaration` tells the LLM when and how to call it. Keep them paired.

2. **Tool descriptions drive routing.** The LLM never sees the function body, only
   the description. A vague description leads to wrong tool selection. Precision matters.

3. **Tools don't have to be read-only.** `calculate_reorder_point` is a pure computation.
   Tools can query databases, write records, send notifications, or call other agents.
   The loop treats them all the same.

4. **The same tool can be called multiple times with different arguments.**
   `get_supplier_quote` was called once for SUPP-A and once for SUPP-B within a single run.
   The agent decides how many times and with what arguments.

5. **Tool results accumulate.** The agent doesn't lose earlier results when it calls a new tool.
   Everything is in the message history, and the LLM reasons over the full accumulated context.
